In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

In [3]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "使用代数符号写出以下方程然后求解。使用以下格式\n\n方程:...\n解:...\n\n",
        ),
        ("human", "{equation_statement}"),
    ]
)
model = ChatOpenAI(temperature=0)
runnable = (
    {"equation_statement": RunnablePassthrough()} | prompt | model | StrOutputParser()
)

print(runnable.invoke("x 的2次方加3等于7"))

方程：x^2 + 3 = 7
解：首先将方程化简为 x^2 = 4
然后求解得 x = ±2
所以方程的解为 x = 2 或 x = -2。


In [5]:
runnable = (
    {"equation_statement": RunnablePassthrough()}
    | prompt
    | model.bind(stop="解")
    | StrOutputParser()
)
print(runnable.invoke("x 的2次方加3等于7"))

方程：x^2 + 3 = 7



In [6]:
function = {
    "name": "solver",
    "description": "构建并求解一个方程",
    "parameters": {
        "type": "object",
        "properties": {
            "equation": {
                "type": "string",
                "description": "方程的代数表达式",
            },
            "solution": {
                "type": "string",
                "description": "方程的解",
            },
        },
        "required": ["equation", "solution"],
    },
}


In [7]:
# 需要 gpt-4 来正确解决这个问题
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "使用代数符号写出以下方程然后求解。",
        ),
        ("human", "{equation_statement}"),
    ]
)
model = ChatOpenAI(model="gpt-4", temperature=0).bind(
    function_call={"name": "solver"}, functions=[function]
)
runnable = {"equation_statement": RunnablePassthrough()} | prompt | model
runnable.invoke("x 的2次方加3等于7")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{\n  "equation": "x^2 + 3 = 7",\n  "solution": "x = ±2"\n}', 'name': 'solver'}}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 100, 'total_tokens': 128}, 'model_name': 'gpt-4', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-06953f94-418c-445f-8dbb-8c56ddb6a25f-0')

In [8]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "获取指定地点的当前天气情况",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "城市和州，例如 San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]


In [9]:
model = ChatOpenAI(model="gpt-3.5-turbo").bind(tools=tools)
model.invoke("旧金山、纽约和洛杉矶的天气怎么样？")

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_rNbtGgKw2KVppZJKHkCln0xD', 'function': {'arguments': '{"location": "San Francisco, CA", "unit": "celsius"}', 'name': 'get_current_weather'}, 'type': 'function'}, {'id': 'call_DM22aRV3aCzEfsdctl9JO9q6', 'function': {'arguments': '{"location": "New York, NY", "unit": "celsius"}', 'name': 'get_current_weather'}, 'type': 'function'}, {'id': 'call_FzGQgwLtCb6bKwzClJQu6mun', 'function': {'arguments': '{"location": "Los Angeles, CA", "unit": "celsius"}', 'name': 'get_current_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 104, 'total_tokens': 188}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-c5ee1e60-0513-42d5-997a-cbbadd4ea0df-0', tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'San Francisco, CA', 'unit': 'celsius'}, 'id': 'call_rNbtGgKw2KVppZJKHkCln0xD'}, {'name': 'get_cu